## A) Keep original pipeline using StandardScaler()

### Ling/NLP

In [1]:
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import StandardScaler

from mlcog import evaluation, models as m
from mlcog import tuning, bootstrap, evaluation 
from mlcog.utils import io

In [2]:
ling_train = pd.read_pickle('../data/features/ling_train.pkl')
X = ling_train['data']
y = ling_train['label']
X_train_2d = np.stack(X.values)
# Normalization
scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train_2d)

USING ROC-AUC AS METRIC

In [3]:
models = m.create_models()
param_grids = m.create_param_grids_()

# model selection and crossvalidation
results = []
for name, model in models.items():
    result = tuning.crossval_(name, model, param_grids[name], X_scaled_train, y, feature_set = 'cv_ling_REP_roc')
    results.append(result)

df_eval_cv = pd.DataFrame(results)
df_eval_cv

Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits


/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Des

Fitting 10 folds for each of 50 candidates, totalling 500 fits


,Model,Sensitivity,Specificity,Roc_auc,Accuracy
0,lr,66.8 (18.4),75.7 (12.5),77.3 (10.8),71.0 (13.2)
1,svm,88.6 (8.6),39.5 (14.9),77.1 (10.0),65.1 (6.9)
2,rf,73.3 (14.6),70.9 (16.8),80.8 (10.8),72.1 (12.3)
3,nn,66.5 (14.0),71.1 (13.5),75.8 (8.1),68.6 (6.4)
4,xgboost,77.9 (9.9),72.5 (20.0),81.7 (10.8),75.1 (10.0)


In [4]:
ling_test = pd.read_pickle('../data/features/ling_test.pkl')

X_test = ling_test['data']
y_test = ling_test['label']
X_test_2d = np.stack(X_test.values)
# Transform test set with scaler object already fit on training data
X_scaled_test = scaler.transform(X_test_2d)

model_map_class = {
        'Logistic Regression': 'lr',
        'SVM': 'svm',
        'Random Forest': 'rf',
        'Neural Network': 'nn',
        'XGBoost': 'xgboost',
    }
best_hyperparams = io.load_best_params(model_map_class, feature_set = 'cv_ling_REP_roc')
best_hyperparams['Random Forest']

,n_estimators,250
,criterion,'gini'
,max_depth,5
,min_samples_split,3
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [5]:
evaluation_bootstrap, probs = bootstrap.fit_and_evaluate_bootstrap_classification(
    best_hyperparams, 
    X_scaled_train, 
    y, 
    X_scaled_test, 
    y_test
)
evaluation_bootstrap_df = pd.DataFrame(evaluation_bootstrap)
df = evaluation_bootstrap_df.round(3)
df[df.columns[1:]] = df[df.columns[1:]] * 100

results_dict = {}
for model_name in df.Model:
    results = evaluation.extract_results_classif_test(df[df.Model == model_name])
    results_dict[model_name] = results
results_dict

{'Logistic Regression': ('74.9 (72.0 - 77.7)',
  '85.0 (80.7 - 89.3)',
  '87.0 (84.7 - 89.2)',
  '80.0 (77.5 - 82.5)'),
 'SVM': ('91.7 (86.8 - 96.7)',
  '38.1 (33.8 - 42.4)',
  '77.3 (74.8 - 79.8)',
  '64.5 (62.6 - 66.4)'),
 'Random Forest': ('69.4 (65.9 - 72.9)',
  '85.0 (80.9 - 89.1)',
  '87.0 (85.1 - 88.9)',
  '77.3 (74.9 - 79.8)'),
 'Neural Network': ('70.9 (65.2 - 76.5)',
  '77.8 (72.6 - 83.0)',
  '82.3 (79.0 - 85.5)',
  '74.4 (70.2 - 78.6)'),
 'XGBoost': ('68.3 (61.3 - 75.3)',
  '82.2 (77.6 - 86.8)',
  '84.3 (81.6 - 87.0)',
  '75.4 (71.5 - 79.2)')}

Pilot

In [8]:
ling_train = pd.read_pickle('../data/features/ling_train.pkl')
X = ling_train['data']
y = ling_train['label']
X_train_2d = np.stack(X.values)
# Normalization
scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train_2d)

### CHANGE feature_set = 
pilot = pd.read_pickle('../data/features/pilot.pkl')
X_test = pilot['data']
y_test = pilot['label']
X_test_2d = np.stack(X_test.values)
# Transform test set with scaler object already fit on training data
X_scaled_test = scaler.transform(X_test_2d)

model_map_class = {
        'Logistic Regression': 'lr',
        'SVM': 'svm',
        'Random Forest': 'rf',
        'Neural Network': 'nn',
        'XGBoost': 'xgboost',
    }
best_hyperparams = io.load_best_params(model_map_class, feature_set = 'cv_ling_REP_roc')

evaluation_bootstrap, probs = bootstrap.fit_and_evaluate_bootstrap_classification(
    best_hyperparams, 
    X_scaled_train, 
    y, 
    X_scaled_test, 
    y_test
)

evaluation_bootstrap_df = pd.DataFrame(evaluation_bootstrap)
df = evaluation_bootstrap_df.round(3)
df[df.columns[1:]] = df[df.columns[1:]] * 100

results_dict = {}
for model_name in df.Model:
    results = evaluation.extract_results_classif_test(df[df.Model == model_name])
    results_dict[model_name] = results
results_dict

{'Logistic Regression': ('72.1 (64.0 - 80.3)',
  '41.2 (30.1 - 52.4)',
  '63.4 (60.4 - 66.4)',
  '60.9 (56.8 - 65.0)'),
 'SVM': ('100.0 (100.0 - 100.0)',
  '1.2 (-1.6 - 4.1)',
  '78.8 (75.3 - 82.3)',
  '64.1 (63.1 - 65.1)'),
 'Random Forest': ('69.3 (60.3 - 78.3)',
  '52.5 (41.5 - 63.5)',
  '65.7 (61.2 - 70.2)',
  '63.2 (59.0 - 67.4)'),
 'Neural Network': ('80.7 (73.5 - 88.0)',
  '33.8 (24.3 - 43.2)',
  '64.1 (59.1 - 69.1)',
  '63.6 (59.6 - 67.7)'),
 'XGBoost': ('78.6 (73.2 - 84.0)',
  '36.2 (26.4 - 46.1)',
  '64.0 (59.9 - 68.1)',
  '63.2 (58.7 - 67.6)')}

External

In [10]:
ling_train = pd.read_pickle('../data/features/ling_train.pkl')
X = ling_train['data']
y = ling_train['label']
X_train_2d = np.stack(X.values)
# Normalization
scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train_2d)

ext = pd.read_pickle('../data/features/ext.pkl')
X_test = ext['data']
y_test = ext['label']

X_test_2d = np.stack(X_test.values)
# Transform test set with scaler object already fit on training data
X_scaled_test = scaler.transform(X_test_2d)

model_map_class = {
        'Logistic Regression': 'lr',
        'SVM': 'svm',
        'Random Forest': 'rf',
        'Neural Network': 'nn',
        'XGBoost': 'xgboost',
    }
best_hyperparams = io.load_best_params(model_map_class, feature_set = 'cv_ling_REP_roc')

evaluation_bootstrap, probs = bootstrap.fit_and_evaluate_bootstrap_classification(
    best_hyperparams, 
    X_scaled_train, 
    y, 
    X_scaled_test, 
    y_test
)

evaluation_bootstrap_df = pd.DataFrame(evaluation_bootstrap)
df = evaluation_bootstrap_df.round(3)
df[df.columns[1:]] = df[df.columns[1:]] * 100

results_dict = {}
for model_name in df.Model:
    results = evaluation.extract_results_classif_test(df[df.Model == model_name])
    results_dict[model_name] = results
results_dict

{'Logistic Regression': ('72.6 (68.2 - 77.0)',
  '75.2 (68.3 - 82.0)',
  '79.9 (76.1 - 83.7)',
  '73.9 (71.0 - 76.8)'),
 'SVM': ('98.9 (97.1 - 100.7)',
  '17.4 (11.8 - 23.0)',
  '79.0 (75.4 - 82.7)',
  '58.1 (56.0 - 60.3)'),
 'Random Forest': ('80.7 (76.3 - 85.2)',
  '70.4 (64.9 - 75.8)',
  '85.0 (84.2 - 85.9)',
  '75.6 (73.4 - 77.7)'),
 'Neural Network': ('72.6 (65.7 - 79.5)',
  '60.0 (48.3 - 71.7)',
  '71.8 (65.5 - 78.1)',
  '66.3 (59.4 - 73.2)'),
 'XGBoost': ('77.0 (71.3 - 82.7)',
  '73.0 (65.9 - 80.0)',
  '83.0 (80.1 - 85.9)',
  '75.0 (71.2 - 78.8)')}

### GPT

In [2]:
with open('../data/features/gpt_train.pkl', 'rb') as f:
    gpt_train = pickle.load(f)

/var/folders/lm/sz7z6d6x725_9nyqv0hzt1jm0000gn/T/ipykernel_59915/1937325124.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  gpt_train = pickle.load(f)


In [3]:
X = gpt_train['data']
y = gpt_train['label']
X_train_2d = np.stack(X.values)
# Normalization
scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train_2d)

from mlcog.utils.io import get_cv_model_path
print(get_cv_model_path("cv_gpt", "lr").resolve())

/Users/marialima/Desktop/GitHub-ML-cog-code/data/cv_eval/cv_gpt/10fcv_lr.pkl


In [4]:
models = m.create_models()
param_grids = m.create_param_grids_()

# model selection and crossvalidation
all_results_gpt = []
for name, model in models.items():
    result_gpt = tuning.crossval_(name, model, param_grids[name], X_scaled_train, y, feature_set = 'cv_gpt_REP_roc')
    all_results_gpt.append(result_gpt)

df_eval_cv_gpt = pd.DataFrame(all_results_gpt)
df_eval_cv_gpt

Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits


/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Des

Fitting 10 folds for each of 50 candidates, totalling 500 fits


,Model,Sensitivity,Specificity,Roc_auc,Accuracy
0,lr,72.2 (14.0),89.8 (9.4),88.7 (5.9),80.7 (7.1)
1,svm,73.5 (12.4),84.8 (12.2),88.4 (6.7),78.9 (7.6)
2,rf,81.4 (14.6),68.6 (15.9),86.0 (8.4),75.3 (9.4)
3,nn,74.6 (14.6),78.6 (12.4),89.1 (6.0),76.5 (7.8)
4,xgboost,83.8 (13.7),75.0 (18.5),88.6 (5.4),79.6 (10.0)


### Using refit = accuracy ***

In [2]:
ling_train = pd.read_pickle('../data/features/ling_train.pkl')
X = ling_train['data']
y = ling_train['label']
X_train_2d = np.stack(X.values)
# Normalization
scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train_2d)

models = m.create_models()
param_grids = m.create_param_grids_()

# model selection and crossvalidation
results = []
for name, model in models.items():
    result = tuning.crossval_(name, model, param_grids[name], X_scaled_train, y, feature_set = 'cv_ling_REP_acc')
    results.append(result)

df_eval_cv = pd.DataFrame(results)
df_eval_cv

Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits


/Users/marialima/Desktop/GitHub-ML-cog-code/venv2/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:609: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv2/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:609: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv2/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:609: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv2/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:609: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima

Fitting 10 folds for each of 50 candidates, totalling 500 fits


,Model,Sensitivity,Specificity,Roc_auc,Accuracy
0,lr,70.4 (10.0),77.3 (9.2),78.6 (6.0),73.6 (6.5)
1,svm,75.0 (14.5),69.5 (14.3),78.1 (5.2),72.2 (8.6)
2,rf,78.8 (16.7),72.1 (13.4),83.5 (8.9),75.3 (9.4)
3,nn,69.4 (18.2),70.7 (17.2),73.4 (12.9),69.9 (7.9)
4,xgboost,79.3 (10.9),78.4 (18.7),86.6 (8.9),78.9 (8.7)


In [ ]:
ling_test = pd.read_pickle('../data/features/ling_test.pkl')

X_test = ling_test['data']
y_test = ling_test['label']
X_test_2d = np.stack(X_test.values)
# Transform test set with scaler object already fit on training data
X_scaled_test = scaler.transform(X_test_2d)

model_map_class = {
        'Logistic Regression': 'lr',
        'SVM': 'svm',
        'Random Forest': 'rf',
        'Neural Network': 'nn',
        'XGBoost': 'xgboost',
    }
best_hyperparams = io.load_best_params(model_map_class, feature_set = 'cv_ling_REP_acc')
best_hyperparams['Random Forest']

In [ ]:
evaluation_bootstrap, probs = bootstrap.fit_and_evaluate_bootstrap_classification(
    best_hyperparams, 
    X_scaled_train, 
    y, 
    X_scaled_test, 
    y_test
)
evaluation_bootstrap_df = pd.DataFrame(evaluation_bootstrap)
df = evaluation_bootstrap_df.round(3)
df[df.columns[1:]] = df[df.columns[1:]] * 100

results_dict = {}
for model_name in df.Model:
    results = evaluation.extract_results_classif_test(df[df.Model == model_name])
    results_dict[model_name] = results
results_dict

In [ ]:
### PILOT

ling_train = pd.read_pickle('../data/features/ling_train.pkl')
X = ling_train['data']
y = ling_train['label']
X_train_2d = np.stack(X.values)
# Normalization
scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train_2d)

### CHANGE feature_set = 
pilot = pd.read_pickle('../data/features/pilot.pkl')
X_test = pilot['data']
y_test = pilot['label']
X_test_2d = np.stack(X_test.values)
# Transform test set with scaler object already fit on training data
X_scaled_test = scaler.transform(X_test_2d)

model_map_class = {
        'Logistic Regression': 'lr',
        'SVM': 'svm',
        'Random Forest': 'rf',
        'Neural Network': 'nn',
        'XGBoost': 'xgboost',
    }
best_hyperparams = io.load_best_params(model_map_class, feature_set = 'cv_ling_REP_acc')

evaluation_bootstrap, probs = bootstrap.fit_and_evaluate_bootstrap_classification(
    best_hyperparams, 
    X_scaled_train, 
    y, 
    X_scaled_test, 
    y_test
)

evaluation_bootstrap_df = pd.DataFrame(evaluation_bootstrap)
df = evaluation_bootstrap_df.round(3)
df[df.columns[1:]] = df[df.columns[1:]] * 100

results_dict = {}
for model_name in df.Model:
    results = evaluation.extract_results_classif_test(df[df.Model == model_name])
    results_dict[model_name] = results
results_dict

In [ ]:
### EXTERNAL 

ling_train = pd.read_pickle('../data/features/ling_train.pkl')
X = ling_train['data']
y = ling_train['label']
X_train_2d = np.stack(X.values)
# Normalization
scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train_2d)

ext = pd.read_pickle('../data/features/ext.pkl')
X_test = ext['data']
y_test = ext['label']

X_test_2d = np.stack(X_test.values)
# Transform test set with scaler object already fit on training data
X_scaled_test = scaler.transform(X_test_2d)

model_map_class = {
        'Logistic Regression': 'lr',
        'SVM': 'svm',
        'Random Forest': 'rf',
        'Neural Network': 'nn',
        'XGBoost': 'xgboost',
    }
best_hyperparams = io.load_best_params(model_map_class, feature_set = 'cv_ling_REP_roc')

evaluation_bootstrap, probs = bootstrap.fit_and_evaluate_bootstrap_classification(
    best_hyperparams, 
    X_scaled_train, 
    y, 
    X_scaled_test, 
    y_test
)

evaluation_bootstrap_df = pd.DataFrame(evaluation_bootstrap)
df = evaluation_bootstrap_df.round(3)
df[df.columns[1:]] = df[df.columns[1:]] * 100

results_dict = {}
for model_name in df.Model:
    results = evaluation.extract_results_classif_test(df[df.Model == model_name])
    results_dict[model_name] = results
results_dict

## B) Use Pipeline() update results
And metric = roc_auc

### Ling-NLP

In [5]:
ling_train = pd.read_pickle('../data/features/ling_train.pkl')
# ling_train.head()

X_train = np.stack(ling_train['data'].values)   
y = ling_train['label'].values

models = m.create_models()
param_grids = m.create_param_grids()

# model selection and crossvalidation
results_ling = []
for name, model in models.items():
    result = tuning.crossval(name, model, param_grids[name], X_train, y, feature_set = 'cv_ling_REP_pipeline')
    results_ling.append(result)

df_eval_cv_ling = pd.DataFrame(results_ling)
df_eval_cv_ling

Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits


/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Des

Fitting 10 folds for each of 50 candidates, totalling 500 fits


,Model,Sensitivity,Specificity,Roc_auc,Accuracy
0,lr,72.8 (19.1),72.0 (18.7),74.8 (15.9),72.2 (13.9)
1,svm,78.5 (18.2),64.5 (16.7),75.1 (15.2),71.7 (12.8)
2,rf,76.1 (14.4),70.7 (15.2),79.0 (8.8),73.4 (11.3)
3,nn,63.2 (12.9),72.1 (10.9),74.8 (8.7),67.4 (7.3)
4,xgboost,77.9 (9.9),75.0 (18.5),78.9 (12.4),76.4 (8.7)


In [6]:
ling_test = pd.read_pickle('../data/features/ling_test.pkl')
model_map_class = {
        'Logistic Regression': 'lr',
        'SVM': 'svm',
        'Random Forest': 'rf',
        'Neural Network': 'nn',
        'XGBoost': 'xgboost',
    }
best_hyperparams = io.load_best_params(model_map_class, feature_set = 'cv_ling_REP_pipeline')
best_hyperparams['Random Forest']

,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_estimators,50
,criterion,'gini'
,max_depth,17
,min_samples_split,3


In [7]:
X_test  = np.stack(ling_test['data'].values)
y_test  = ling_test['label'].values

evaluation_bootstrap, probs = bootstrap.fit_and_evaluate_bootstrap_classification(
    best_hyperparams, 
    X_train, 
    y, 
    X_test, 
    y_test
)
evaluation_bootstrap_df = pd.DataFrame(evaluation_bootstrap)
df = evaluation_bootstrap_df.round(3)
df[df.columns[1:]] = df[df.columns[1:]] * 100

results_dict = {}
for model_name in df.Model:
    results = evaluation.extract_results_classif_test(df[df.Model == model_name])
    results_dict[model_name] = results
results_dict

{'Logistic Regression': ('70.6 (68.0 - 73.1)',
  '89.4 (87.2 - 91.7)',
  '88.2 (86.7 - 89.7)',
  '80.1 (78.8 - 81.5)'),
 'SVM': ('76.0 (65.4 - 86.6)',
  '77.5 (61.4 - 93.6)',
  '88.8 (87.3 - 90.3)',
  '76.8 (71.9 - 81.6)'),
 'Random Forest': ('69.1 (64.2 - 74.0)',
  '81.4 (78.3 - 84.5)',
  '84.2 (82.3 - 86.1)',
  '75.4 (72.7 - 78.0)'),
 'Neural Network': ('70.3 (65.1 - 75.5)',
  '78.1 (73.3 - 82.8)',
  '83.4 (80.5 - 86.3)',
  '74.2 (70.0 - 78.4)'),
 'XGBoost': ('72.0 (67.2 - 76.8)',
  '83.3 (80.5 - 86.1)',
  '85.6 (84.0 - 87.1)',
  '77.7 (75.2 - 80.3)')}

Pilot

In [8]:
pilot = pd.read_pickle('../data/features/pilot.pkl')
X_test  = np.stack(pilot['data'].values)
y_test  = pilot['label'].values

evaluation_bootstrap, probs = bootstrap.fit_and_evaluate_bootstrap_classification(
    best_hyperparams, 
    X_train, 
    y, 
    X_test, 
    y_test
)
evaluation_bootstrap_df = pd.DataFrame(evaluation_bootstrap)
df = evaluation_bootstrap_df.round(3)
df[df.columns[1:]] = df[df.columns[1:]] * 100

results_dict = {}
for model_name in df.Model:
    results = evaluation.extract_results_classif_test(df[df.Model == model_name])
    results_dict[model_name] = results
results_dict["Random Forest"]

{'Logistic Regression': ('54.3 (48.8 - 59.8)',
  '56.2 (44.9 - 67.6)',
  '53.3 (49.4 - 57.2)',
  '55.0 (50.8 - 59.2)'),
 'SVM': ('61.4 (51.7 - 71.1)',
  '46.2 (31.0 - 61.5)',
  '55.6 (52.1 - 59.2)',
  '55.9 (52.8 - 59.0)'),
 'Random Forest': ('69.3 (58.2 - 80.3)',
  '55.0 (45.4 - 64.6)',
  '67.7 (61.2 - 74.2)',
  '64.1 (55.4 - 72.8)'),
 'Neural Network': ('77.9 (70.5 - 85.3)',
  '36.2 (28.4 - 44.1)',
  '64.4 (59.3 - 69.5)',
  '62.7 (59.7 - 65.7)'),
 'XGBoost': ('81.4 (75.4 - 87.4)',
  '40.0 (31.8 - 48.2)',
  '64.7 (60.4 - 69.1)',
  '66.4 (61.5 - 71.3)')}

External

In [10]:
ext = pd.read_pickle('../data/features/ext.pkl')
X_test  = np.stack(ext['data'].values)
y_test  = ext['label'].values

evaluation_bootstrap, probs = bootstrap.fit_and_evaluate_bootstrap_classification(
    best_hyperparams, 
    X_train, 
    y, 
    X_test, 
    y_test
)
evaluation_bootstrap_df = pd.DataFrame(evaluation_bootstrap)
df = evaluation_bootstrap_df.round(3)
df[df.columns[1:]] = df[df.columns[1:]] * 100

results_dict = {}
for model_name in df.Model:
    results = evaluation.extract_results_classif_test(df[df.Model == model_name])
    results_dict[model_name] = results
results_dict["Random Forest"]

('79.6 (76.0 - 83.2)',
 '68.1 (61.9 - 74.4)',
 '82.7 (80.2 - 85.3)',
 '73.9 (70.7 - 77.1)')

### GPT

In [11]:
with open('../data/features/gpt_train.pkl', 'rb') as f:
    gpt_train = pickle.load(f)

X_gpt_train = np.stack(gpt_train['data'].values)   
y_gpt_train = gpt_train['label'].values

models = m.create_models()
param_grids = m.create_param_grids()

# model selection and crossvalidation
all_results_gpt = []
for name, model in models.items():
    result_gpt = tuning.crossval(name, model, param_grids[name], X_gpt_train, y_gpt_train, feature_set = 'cv_gpt_REP_pipeline')
    all_results_gpt.append(result_gpt)

df_eval_cv_gpt = pd.DataFrame(all_results_gpt)
df_eval_cv_gpt

Fitting 10 folds for each of 50 candidates, totalling 500 fits


/var/folders/lm/sz7z6d6x725_9nyqv0hzt1jm0000gn/T/ipykernel_59915/2879083581.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  gpt_train = pickle.load(f)


Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits
Fitting 10 folds for each of 50 candidates, totalling 500 fits


/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Desktop/GitHub-ML-cog-code/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/marialima/Des

Fitting 10 folds for each of 50 candidates, totalling 500 fits


,Model,Sensitivity,Specificity,Roc_auc,Accuracy
0,lr,75.4 (16.0),86.2 (11.8),87.6 (5.7),80.7 (8.8)
1,svm,79.2 (13.9),82.5 (11.5),87.1 (5.6),80.7 (8.1)
2,rf,85.0 (12.9),73.8 (19.7),85.6 (7.5),79.6 (9.1)
3,nn,75.7 (14.4),79.8 (11.4),88.7 (6.1),77.7 (7.2)
4,xgboost,83.9 (12.0),77.5 (15.6),85.0 (7.8),80.8 (9.7)
